In [1]:
import requests
import pandas as pd
import json
from datetime import datetime

In [2]:
# Configuration
GAMMA_URL = "https://gamma-api.polymarket.com/markets"

In [3]:
def fetch_active_polymarkets():
    """
    Fetches active markets from Polymarket Gamma API.
    Note: Using a high limit to get as many as possible; 
    in a production script, you would handle pagination.
    """
    params = {
        "active": "true",
        "closed": "false",
        "limit": 500,        # Max limit per request
        "order": "volume",   # Start with most liquid
        "ascending": "false"
    }
    
    response = requests.get(GAMMA_URL, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return []

In [4]:
# 1. Fetch Data
raw_data = fetch_active_polymarkets()

In [5]:
# 2. Define our Keyword Lists for "Slicing"
DIRECT_FINANCE_KEYS = [
    'fed', 'interest', 'rate', 'cpi', 'inflation', 'gdp', 'recession', 
    'earnings', 'nvidia', 'apple', 'tesla', 'bitcoin', 'eth', 'crypto', 
    'market cap', 'unemployment', 'yield', 'treasury'
]

In [6]:
INDIRECT_FINANCE_KEYS = [
    'trump', 'biden', 'election', 'war', 'china', 'tariffs', 'israel', 
    'ukraine', 'russia', 'strike', 'regulation', 'sec', 'supreme court',
    'protest', 'houthis', 'middle east', 'nuclear'
]

In [7]:
processed_list = []

In [8]:
for m in raw_data:
    title = m.get('question', '').lower()
    description = m.get('description', '').lower()
    full_text = title + " " + description
    
    # Determine Category
    category = "General"
    if any(k in full_text for k in DIRECT_FINANCE_KEYS):
        category = "Direct Finance"
    elif any(k in full_text for k in INDIRECT_FINANCE_KEYS):
        category = "Indirect/Geopolitical"
    
    # Extract Probability (Price)
    # Outcome prices are stored as strings in a list [Yes_Price, No_Price]
    try:
        outcome_prices = json.loads(m.get('outcomePrices', '["0", "0"]'))
        prob_yes = float(outcome_prices[0]) * 100
    except:
        prob_yes = None

    processed_list.append({
        'Question': m.get('question'),
        'Category': category,
        'Probability_%': prob_yes,
        'Total_Volume': float(m.get('volume', 0)),
        'Liquidity': float(m.get('liquidity', 0)),
        'End_Date': m.get('endDate'),
        'Market_ID': m.get('id')
    })

df = pd.DataFrame(processed_list)

In [9]:
# 4. Analyze the Dataset
print(f"Total Active Markets Fetched: {len(df)}")
print("-" * 30)
print("Breakdown by Category:")
print(df['Category'].value_counts())
print("-" * 30)

# 5. Filter for "Most Liquid" Finance Markets
# These are your primary 'Factors'
top_finance = df[df['Category'] == 'Direct Finance'].sort_values(by='Total_Volume', ascending=False)

print("Top 5 Most Liquid Finance Events (Potential Alpha Factors):")
print(top_finance[['Question', 'Total_Volume', 'Probability_%']].head(5))

# 6. Filter for "Most Liquid" Geopolitical Markets
# These are your 'Macro Mood' indicators
top_geo = df[df['Category'] == 'Indirect/Geopolitical'].sort_values(by='Total_Volume', ascending=False)

print("\nTop 5 Most Liquid Geopolitical Events (Macro Bias Factors):")
print(top_geo[['Question', 'Total_Volume', 'Probability_%']].head(5))

Total Active Markets Fetched: 500
------------------------------
Breakdown by Category:
Category
General                  242
Indirect/Geopolitical    131
Direct Finance           127
Name: count, dtype: int64
------------------------------
Top 5 Most Liquid Finance Events (Potential Alpha Factors):
                                              Question  Total_Volume  \
161              US strikes Iran by February 28, 2026?  9.720946e+06   
164  Will Crystal Palace win the 2025–26 English Pr...  9.716745e+06   
373  Will Fulham win the 2025–26 English Premier Le...  9.401014e+06   
402     Will Benfica win the 2025–26 Champions League?  9.357448e+06   
449      Will Athletic Bilbao win the 2025–26 La Liga?  9.292397e+06   

     Probability_%  
161          30.50  
164           0.15  
373           0.15  
402           0.65  
449           0.10  

Top 5 Most Liquid Geopolitical Events (Macro Bias Factors):
                                              Question  Total_Volume  \
16   Wi

In [10]:
import requests
import pandas as pd
import json

# Gamma API endpoint for markets
# We filter for 'active' and 'limit=100' for a clean sample
URL = "https://gamma-api.polymarket.com/markets?active=true&closed=false&limit=100"

response = requests.get(URL)
markets = response.json()

# Let's grab the first market to see ALL available attributes
sample_market = markets[0]
attributes = list(sample_market.keys())

print("--- FULL LIST OF ATTRIBUTES FOR EACH BET ---")
for attr in attributes:
    print(f"-> {attr}")
print(f"\nTotal Attributes per market: {len(attributes)}")

--- FULL LIST OF ATTRIBUTES FOR EACH BET ---
-> id
-> question
-> conditionId
-> slug
-> resolutionSource
-> endDate
-> liquidity
-> startDate
-> image
-> icon
-> description
-> outcomes
-> outcomePrices
-> volume
-> active
-> closed
-> marketMakerAddress
-> createdAt
-> updatedAt
-> new
-> featured
-> submitted_by
-> archived
-> resolvedBy
-> restricted
-> groupItemTitle
-> groupItemThreshold
-> questionID
-> enableOrderBook
-> orderPriceMinTickSize
-> orderMinSize
-> volumeNum
-> liquidityNum
-> endDateIso
-> startDateIso
-> hasReviewedDates
-> volume24hr
-> volume1wk
-> volume1mo
-> volume1yr
-> clobTokenIds
-> umaBond
-> umaReward
-> volume24hrClob
-> volume1wkClob
-> volume1moClob
-> volume1yrClob
-> volumeClob
-> liquidityClob
-> acceptingOrders
-> negRisk
-> negRiskMarketID
-> negRiskRequestID
-> events
-> ready
-> funded
-> acceptingOrdersTimestamp
-> cyom
-> competitive
-> pagerDutyNotificationEnabled
-> approved
-> rewardsMinSize
-> rewardsMaxSpread
-> spread
-> oneDayPriceCh

In [11]:
import requests
import pandas as pd
import json

def fetch_all_finance_markets():
    # Polymarket Gamma API Endpoint
    url = "https://gamma-api.polymarket.com/markets"
    
    # We will fetch a large batch of active markets
    params = {
        "active": "true",
        "closed": "false",
        "limit": 500,  # Max allowed per request
        "order": "volume",
        "ascending": "false"
    }
    
    print("Fetching data from Polymarket Gamma API...")
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None

    all_markets = response.json()
    
    # Comprehensive Finance/Macro Keywords
    FINANCE_KEYWORDS = [
        'fed', 'interest', 'rate', 'cpi', 'inflation', 'gdp', 'recession', 
        'yield', 'treasury', 'unemployment', 'payroll', 'stock', 'nvidia', 
        'apple', 'tesla', 'bitcoin', 'btc', 'eth', 'crypto', 'coinbase', 
        'sec', 'etf', 'halving', 'sp500', 'nasdaq', 'dow jones', 'economy'
    ]

    finance_data = []

    for m in all_markets:
        full_text = f"{m.get('question', '')} {m.get('description', '')}".lower()
        
        # Check if it's a finance-related bet
        if any(key in full_text for key in FINANCE_KEYWORDS):
            
            # Extract Odds (Outcome Prices are strings in a list like ["0.45", "0.55"])
            prices = m.get('outcomePrices')
            if isinstance(prices, str):
                prices = json.loads(prices)
            
            # Note: Index 0 is usually 'YES', Index 1 is 'NO'
            yes_price = float(prices[0]) if prices and len(prices) > 0 else 0
            no_price = float(prices[1]) if prices and len(prices) > 1 else 0

            # Map ALL available attributes into a clean dictionary
            finance_data.append({
                'Market_ID': m.get('id'),
                'Question': m.get('question'),
                'Status_Active': m.get('active'),
                'Yes_Price_Probability': yes_price,
                'No_Price_Probability': no_price,
                'Total_Volume': float(m.get('volume', 0)),
                'Total_Liquidity': float(m.get('liquidity', 0)),
                'Category': m.get('category'),
                'Group_ID': m.get('group_id'),
                'End_Date': m.get('endDate'),
                'Last_Updated': m.get('updatedAt'),
                # CLOB Token IDs are critical for real-time tracking later
                'CLOB_Token_IDs': m.get('clobTokenIds'), 
                'Description': m.get('description'),
                'Icon_URL': m.get('icon'),
                'Rewards_Max': m.get('rewardsMax'),
                'Spread': m.get('spread')
            })

    # Create DataFrame
    df = pd.DataFrame(finance_data)
    
    # Save to CSV
    filename = "polymarket_finance_full_metadata.csv"
    df.to_csv(filename, index=False)
    print(f"Success! Found {len(df)} finance-related markets.")
    print(f"Data saved to: {filename}")
    
    return df

# Execute and view
finance_df = fetch_all_finance_markets()

# Quick look at the top liquid finance bets
if finance_df is not None:
    print("\nTop 5 Finance Bets by Volume:")
    print(finance_df[['Question', 'Total_Volume', 'Yes_Price_Probability']].head())

Fetching data from Polymarket Gamma API...
Success! Found 165 finance-related markets.
Data saved to: polymarket_finance_full_metadata.csv

Top 5 Finance Bets by Volume:
                                            Question   Total_Volume  \
0  Will MrBeast's next video get less than 25 mil...    9997.647194   
1  Will José Luna win the 2026 Peruvian president...    9996.070007   
2  Will Bitcoin dip to $55,000 by December 31, 2026?  997281.680144   
3  Will a dozen eggs cost between $2.75–3.00 in J...    9970.793091   
4  Will Coinbase say "Staking" or "Stake" during ...    9957.574467   

   Yes_Price_Probability  
0                 0.0375  
1                 0.0050  
2                 0.7050  
3                 0.0625  
4                 0.7250  


In [12]:
import requests
import pandas as pd

# Gamma API uses 'tags' to organize the sections you see in the UI
def fetch_polymarket_by_category(tag_slug):
    # Tag slugs usually match the UI: 'stocks', 'crypto', 'economics', 'politics'
    url = f"https://gamma-api.polymarket.com/markets?tag={tag_slug}&active=true"
    
    print(f"Slicing Polymarket: Section '{tag_slug}'...")
    response = requests.get(url)
    
    if response.status_code != 200:
        return None

    data = response.json()
    extracted = []
    
    for m in data:
        # Extract the 'Yes' price (implied probability)
        # OutcomePrices is a list of strings: ["YesPrice", "NoPrice"]
        try:
            prices = eval(m.get('outcomePrices', '["0", "0"]'))
            prob = float(prices[0])
        except:
            prob = 0

        extracted.append({
            'Section': tag_slug,
            'Ticker_Event': m.get('question'),
            'Probability': prob,
            'Daily_Vol': m.get('volume24hr', 0),
            'Total_Vol': m.get('volume', 0),
            'Liquidity': m.get('liquidity', 0),
            'Market_ID': m.get('id'),
            'Token_ID_Yes': eval(m.get('clobTokenIds', '[]'))[0] if m.get('clobTokenIds') else None
        })
    
    return pd.DataFrame(extracted)

# --- THE QUANT WORKFLOW ---
# We will "Slice" the sections exactly as seen in your screenshot
sections = ['stocks', 'crypto', 'economics', 'business']
all_finance_data = []

for section in sections:
    df_section = fetch_polymarket_by_category(section)
    if df_section is not None:
        all_finance_data.append(df_section)

# Combine into one master "Factor Catalog"
master_df = pd.concat(all_finance_data, ignore_index=True)

# Save for your Professor
master_df.to_csv("polymarket_factor_catalog.csv", index=False)

print("\n--- Data Ready for Office Hours ---")
print(master_df.groupby('Section')['Ticker_Event'].count())

Slicing Polymarket: Section 'stocks'...
Slicing Polymarket: Section 'crypto'...
Slicing Polymarket: Section 'economics'...
Slicing Polymarket: Section 'business'...

--- Data Ready for Office Hours ---
Section
business     20
crypto       20
economics    20
stocks       20
Name: Ticker_Event, dtype: int64
